# lasso_and_ridge_shrinkage

> 对应代码：`EconometricsCode/code_in_notes/Chap.5/lasso_and_ridge_shrinkage.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.5`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.5")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先做常规的环境清理，然后读入 hcw 数据集——这是一个包含香港及其他亚太经济体序列的时间序列数据，并用 tsset 把 time 声明为时间变量，这样后面的 tsline 等时间序列命令才能识别时间轴。

In [ ]:
%%stata
clear
set more off
use "../datasets/hcw.dta"
tsset time

我们首先估计 Lasso 线性回归：被解释变量是 HongKong，解释变量为从 Australia 到 Thailand 的所有变量（即数据集中按变量表顺序排列的各经济体序列），if _n<=18 表示只用前 18 期观测做训练。rseed(5) 固定交叉验证随机分折的种子以保证结果可复现；sel(cv) 指定用交叉验证（cross-validation）选择调节参数 λ，即在使交叉验证均方误差最小的 λ 处停止。Lasso 使用 L1 惩罚，其几何特性会把不重要变量的系数精确压缩到 0，从而同时完成变量选择与收缩估计。

In [ ]:
%%stata
// Lasso
lasso linear HongKong Australia-Thailand if _n<=18, rseed(5) sel(cv)

coefpath 画出系数路径图：横轴是惩罚参数 λ（通常取对数），纵轴是各变量系数。我们从中可以直观看到，随着 λ 减小、惩罚放松，各变量系数从 0 开始逐一“进入”模型的过程。

In [ ]:
%%stata
coefpath

cvplot 画出交叉验证曲线，即不同 λ 下的交叉验证均方误差，并标出使误差最小的 λ 以及一倍标准误规则对应的 λ*，帮助我们理解交叉验证是如何在拟合优度与模型复杂度之间做权衡的。

In [ ]:
%%stata
cvplot

etable 把刚才估计的 Lasso 结果整理成规范的结果表格，便于报告与比较。

In [ ]:
%%stata
etable

lassocoef 列出在选定的 λ 下各变量的系数估计（包括被压缩为 0 的变量），让我们看清楚 Lasso 到底保留了哪些变量、系数被收缩了多少。

In [ ]:
%%stata
lassocoef

lassoinfo 报告本次 Lasso 估计的概要信息，如所选 λ 的值、交叉验证误差、被选入的非零系数个数等。

In [ ]:
%%stata
lassoinfo

接下来我们估计岭回归（ridge regression）。Stata 中没有单独的 ridge 命令，而是通过 elasticnet 并把弹性网混合参数设为 alpha(0) 来实现：弹性网惩罚是 α·L1 与 (1−α)·L2 的线性组合，α=0 时退化为纯 L2 惩罚，即岭回归。岭回归会把所有系数向 0 收缩，但一般不会把系数精确压到 0，因此它只做收缩、不做变量选择，且在高度相关的解释变量之间倾向于平均分配系数。其余选项 rseed(5) 与 sel(cv) 的含义与上面 Lasso 相同。

In [ ]:
%%stata
// ridge
elasticnet linear HongKong Australia-Thailand if _n<=18,  rseed(5) sel(cv) alpha(0)

coefpath 画出岭回归的系数路径。与 Lasso 路径对比可以发现：岭回归的路径是平滑收缩的曲线，所有系数同时缓慢趋向 0，而不会出现 Lasso 那样系数被逐一“压成 0”的折角。

In [ ]:
%%stata
coefpath

cvplot 画出岭回归的交叉验证曲线，展示交叉验证误差随 λ 变化的情况及最优 λ 的位置。

In [ ]:
%%stata
cvplot

etable 输出岭回归结果的规范表格。

In [ ]:
%%stata
etable

lassocoef 列出岭回归在选定 λ 下的系数。可以看到岭回归保留了全部变量，只是把所有系数都向 0 做了不同程度的收缩。

In [ ]:
%%stata
lassocoef

lassoinfo 报告岭回归估计的概要信息，包括选定的 λ 与相应的交叉验证误差。

In [ ]:
%%stata
lassoinfo

下面进入预测比较部分。我们先重新估计一遍交叉验证的 Lasso（结果与前文一致），为接下来生成预测值做准备。

In [ ]:
%%stata
// adpative lasso
lasso linear HongKong Australia-Thailand if _n<=18, rseed(5) sel(cv)

predict 生成 Lasso 的样本内拟合值，存入新变量 HongKong_lasso。注意 Lasso 的系数本身因惩罚而收缩、有偏，直接用于预测时收缩带来的偏差仍然存在。

In [ ]:
%%stata
predict HongKong_lasso

加上 post 选项的 predict 生成 Post-Lasso 预测值：先由 Lasso 选出非零系数的变量集合，再仅用这个子集做普通最小二乘（OLS）重新估计并预测。这一步“重估计”去掉了 Lasso 对存活系数的收缩偏差，是 Belloni 和 Chernozhukov 推荐的做法，在保留变量选择功能的同时改善估计与预测精度。

In [ ]:
%%stata
predict HongKong_lasso_post, post

我们为两个预测序列加上变量标签，方便稍后图例中区分 Lasso 与 Post-Lasso 两种预测。

In [ ]:
%%stata
label variable HongKong_lasso "Lasso"
label variable HongKong_lasso_post "PostLasso"

这里估计自适应 Lasso（adaptive Lasso）。sel(adaptive) 表示用自适应方式施加惩罚：先做一次初始估计（由交叉验证的 Lasso 得到），再用各变量初始系数绝对值的倒数作为权重构造惩罚项，即对每个系数施加 λ·wⱼ·|βⱼ| 的差异化惩罚。直觉上，初始估计大的变量惩罚轻、初始估计小的变量惩罚重，这使自适应 Lasso 具有“神谕性质”（oracle property）：在大样本下能以趋于 1 的概率选出正确的变量集合，且非零系数的估计渐近有效，从而克服了普通 Lasso 收缩不一致的问题。

In [ ]:
%%stata
lasso linear HongKong Australia-Thailand if _n<=18, rseed(5) sel(adaptive)

lassocoef 列出自适应 Lasso 的系数估计，我们可以与普通 Lasso 的结果对比，看自适应惩罚对变量选择和系数大小的影响。

In [ ]:
%%stata
lassocoef

predict 生成自适应 Lasso 的拟合（预测）值，存入 HongKong_adalasso。

In [ ]:
%%stata
predict HongKong_adalasso

为自适应 Lasso 的预测序列加上变量标签，供图例使用。

In [ ]:
%%stata
label variable HongKong_adalasso "Adaptive Lasso"

我们用 twoway 把真实序列 HongKong 与三种预测序列（Lasso、Post-Lasso、自适应 Lasso）画在同一张时间序列图上，实线为真实值，其余分别以虚线、点线、点划线区分。通过对比各条曲线对真实序列的贴合程度，我们可以直观评价不同收缩方法的样本内预测表现。

In [ ]:
%%stata
twoway (tsline HongKong, lcolor(black)) (tsline HongKong_lasso, lcolor(black) lpattern(dash)) (tsline HongKong_lasso_post, lcolor(black) lpattern(dot)) (tsline HongKong_adalasso, lcolor(black) lpattern(dash_dot)) if _n<=18

最后我们演示如何手工实现 Post-Lasso。先重新跑一遍交叉验证 Lasso，此时 e(post_sel_vars) 中保存了 Lasso 选出的非零系数变量名单。

In [ ]:
%%stata
lasso linear HongKong Australia-Thailand if _n<=18, rseed(5) sel(cv)

我们把 e(post_sel_vars) 的内容存入局部宏 post_vars，便于下一步引用。

In [ ]:
%%stata
local post_vars=e(post_sel_vars)

用 reg 对 Lasso 选出的变量子集做普通 OLS 回归，这就是 Post-Lasso 的第二步：在选中的模型上做无惩罚的重估计，其估计结果与前面 predict ..., post 所隐含的重估计一致。

In [ ]:
%%stata
reg `post_vars'